# Overview

Goal of this notebook is to create the train/test/holdout splits for the project.

For the splits themselves, we want to pull one row per building. In this data basically every building shows up multiple times, so we only want a single instance of that building to train/test on.

In [38]:
import polars as pl
from src.helpers.eda import get_column_value_frequency

data_directory = "../../data/energy"
df = pl.read_parquet(f"{data_directory}/s_energy.parquet")

In [39]:
df

OSEBuildingID,DataYear,BuildingName,BuildingType,Neighborhood,YearBuilt,NumberofFloors,NumberofBuildings,PropertyGFABuildings,EPAPropertyType,EUI,EUI_WN,Year2025
i64,i64,str,str,str,i64,i64,i64,str,str,str,str,bool
2,2025,"""Paramount Hotel""","""NonResidential""","""EAST""",1996,11,1,"""88,502""","""Hotel""","""69.3""","""69""",true
3,2025,"""Westin Hotel (Parent Building)""","""NonResidential""","""LAKE UNION""",1969,41,3,"""759,392""","""Hotel""","""89.9""","""89.9""",true
5,2025,"""Hotel Max""","""NonResidential""","""LAKE UNION""",1926,10,1,"""61,320""","""Hotel""","""83.6""","""83.2""",true
8,2025,"""Warwick Seattle Hotel""","""NonResidential""","""LAKE UNION""",1980,18,1,"""113,580""","""Hotel""","""86.1""","""85.7""",true
9,2025,"""West Precinct (Seattle Police)""","""NonResidential""","""LAKE UNION""",1999,2,1,"""60,090""","""Police Station""","""172.1""","""171.7""",true
…,…,…,…,…,…,…,…,…,…,…,…,…
50038,2015,"""Salmon Bay Marine Center West""","""NonResidential""","""MAGNOLIA / QUEEN ANNE""",2014,2,1,"""25,532""","""Mixed Use Property""","""26.79999924""","""29.10000038""",false
50049,2015,"""PACIFIC CENTER CONDOMINIUM""","""Multifamily LR (1-4)""","""DOWNTOWN""",2000,4,1,"""44,650""","""Multifamily LR (1-4)""","""25.60000038""","""27.79999924""",false
50055,2015,"""NORA (IDENTITY APTS 4123)""","""Multifamily MR (5-9)""","""NORTHEAST""",2014,7,1,"""42,393""","""Multifamily MR (5-9)""","""48.59999847""","""55.09999847""",false


In [52]:
with pl.Config(tbl_rows=-1):
    print(
        df
        .select("PropertyGFABuildings")
        .with_columns((pl.col("PropertyGFABuildings").str.replace(pattern=",", value="").alias("PropertyGFABuildingsFIXED")))
        .head(100)
    )

shape: (100, 2)
┌──────────────────────┬───────────────────────────┐
│ PropertyGFABuildings ┆ PropertyGFABuildingsFIXED │
│ ---                  ┆ ---                       │
│ str                  ┆ str                       │
╞══════════════════════╪═══════════════════════════╡
│ 88,502               ┆ 88502                     │
│ 759,392              ┆ 759392                    │
│ 61,320               ┆ 61320                     │
│ 113,580              ┆ 113580                    │
│ 60,090               ┆ 60090                     │
│ 102,761              ┆ 102761                    │
│ 163,984              ┆ 163984                    │
│ 62,216               ┆ 62216                     │
│ 133,884              ┆ 133884                    │
│ 272,015              ┆ 272015                    │
│ 65,009               ┆ 65009                     │
│ 295,511              ┆ 295511                    │
│ 67,390               ┆ 67390                     │
│ 355,000              ┆ 35500

## Examine grain to motivate future methods

In [40]:
(
    df
    .group_by("OSEBuildingID")
    .len()
    .sort("len", descending=True)
)

OSEBuildingID,len
i64,u32
27829,11
19671,11
423,11
27707,11
19668,11
…,…
50956,1
52022,1
52031,1


In [41]:
(
    df
    .group_by("OSEBuildingID", "DataYear")
    .len()
    .sort("len", descending=True)
)

OSEBuildingID,DataYear,len
i64,i64,u32
20017,2021,1
27264,2017,1
21774,2019,1
49710,2025,1
26898,2023,1
…,…,…
50148,2023,1
25743,2015,1
20071,2018,1


## Set it all up

Plan:

In [47]:
final_year_set = df.filter(pl.col("DataYear") == 2025)

buildings = (
    final_year_set
    .select("OSEBuildingID")
    .unique()
    .shape[0]
)
print(f"\nNumber of Buildings in Dataset: {buildings}\n")


Number of Buildings in Dataset: 3462



In [45]:
# Goal:
df_shuffled = (
    df
    .filter(
        pl.int_range(0, pl.len())
        .shuffle(seed=101)
        .over("OSEBuildingID") == 0
    )
)

df_shuffled

OSEBuildingID,DataYear,BuildingName,BuildingType,Neighborhood,YearBuilt,NumberofFloors,NumberofBuildings,PropertyGFABuildings,EPAPropertyType,EUI,EUI_WN,Year2025
i64,i64,str,str,str,i64,i64,i64,str,str,str,str,bool
830,2025,"""Cascade Public Media (Formerly…","""NonResidential""","""EAST""",2004,4,1,"""45,294""","""Office""","""29.1""","""28.9""",true
19537,2025,"""Alki Elementary School & Commu…","""School""","""SOUTHWEST""",1954,2,3,"""51,489""","""K-12 School""","""14.2""","""14.2""",true
22701,2025,"""Victory Studios""","""NonResidential""","""MAGNOLIA/QUEEN ANNE""",1968,2,1,"""25,946""","""Other""","""4,474.8""","""4,473.8""",true
24371,2025,"""Bb House""","""Multifamily LR (1-4)""","""EAST""",2024,5,0,"""21,072""","""Multifamily Housing""","""25.5""","""25.3""",true
24514,2025,"""Oak Tree Plaza - Supermarket""","""NonResidential""","""NORTHWEST""",1986,1,1,"""45,155""","""Supermarket/Grocery Store""","""193.4""","""193.4""",true
…,…,…,…,…,…,…,…,…,…,…,…,…
24351,2016,"""1964 4TH AVE S""","""NonResidential""","""GREATER DUWAMISH""",1916,3,2,"""71,283""","""Non-Refrigerated Warehouse""","""34.90000153""","""37.20000076""",false
27016,2016,"""PACIFIC GEORGETOWN CENTER LLC""","""NonResidential""","""GREATER DUWAMISH""",1969,1,1,"""35,017""","""Office""","""43.20000076""","""46.40000153""",false
27455,2016,"""Queen Anne Exchange""","""Multifamily LR (1-4)""","""MAGNOLIA / QUEEN ANNE""",1921,3,0,"""23,040""","""Distribution Center""","""18.29999924""","""20.5""",false


In [43]:
(
    df_shuffled
    .group_by("OSEBuildingID")
    .len()
    .sort("len", descending=True)
)

OSEBuildingID,len
i64,u32
49703,1
24560,1
50638,1
414,1
20868,1
…,…
50715,1
50694,1
23163,1


In [44]:
get_column_value_frequency(df_shuffled, "DataYear")

DataYear,count
i64,u32
2015,562
2016,2567
2017,26
2018,146
2019,26
…,…
2021,3
2022,1
2023,5
